# Antiferromagnetism in the La$_{2}$CuO$_{4}$ one-band model

In this tutorial, the objective is to perform a DMFT calculation in an **ordered phase**. Again, we take the example of La$_2$CuO$_4$ which in reality is an antiferromagnet at low temperature. We will use the one-band model derived before. We emphasize that this is treated as a toy-model and doesn't capture all the important physical elements required to correctly characterize La$_2$CuO$_4$. For a better model (see [Tutorial 05](05as-la2cuo4.ipynb)). However, by using an artificially large on-site Coulomb repulsion $U$, we can mimic the insulating behavior and the antiferromagnetic state in this toy-model.

This example will allow us to introduce:
- The construction of **super lattices** from the DFT extracted tight-binding Hamiltonians, and
- The **non-trivial spin mapping** allowed by the embedding object, enabling solving a single impurity problem to capture the physics of both sublattices in the antiferromagnetic state.

<div style="text-align: center;">
<img src="attachment:afm_cell.png" alt="afm_cell" width="750"/>
</div>

## Tight-binding model on a super lattice

The solution we are looking for is antiferromagnetic. In particular, this solution involves a non-zero symmetry-breaking order parameter: the **staggered magnetization** defined as
$$ M = \sum_i (-1)^i \langle n_{i\uparrow} - n_{i\downarrow} \rangle, $$
where $(-1)^i$ is a phase between in-plane nearest neighbors. In other words, this order parameter breaks the translational invariance of the system, which now has a sublattice $A$ polarized in the $\uparrow$ spin and another sublattice $B$ polarized in the $\downarrow$ spin.

In order to allow DMFT to capture this solution, we need to consider the **two sublattices** in some way. To do this, we will double the unit cell with respect to the paramagnetic phase to allow for the ordered phase to emerge.

There are two ways to achieve this:
1) Perform a DFT calculation in the supercell and downfold onto the 2 Cu-$d_{x^{2}-y^{2}}$ bands near the Fermi level.
2) Use the Wannier model developed from the DFT calculation in the unit cell and fold the hopping matrix elements onto the desired super lattice.

In this tutorial, we will take the second route to highlight additional features of the TRIQS. First, load the one-body element corresponding to the one-band model for La$_2$CuO$_4$ used in the [Tutorial 02](02-dmft-lco-1band.ipynb).

In [ ]:
import triqs_modest as modest

seedname = 'data/mlwf/lco'
spin_kind = "NonPolarized"

shells = [modest.AtomicOrbs(l=2,dim=1,dft_idx=0,cls_idx=0)]
obe = modest.one_body_elements_from_wannier90(seedname, spin_kind, shells)

From this obe, we can construct a super lattice using `modest.Superlattice` and produce a new obe on this super lattice using `modest.fold`. This function takes two arguments: first the super lattice vectors that will define the new cell and second the sites from the original lattice that should be considered inside the new unit cell. These arguments have to be expressed in the basis lattice vectors of the original lattice. To find these vectors, we inspect the Wannier90 output that generated the model using

In [ ]:
!grep -A 3 Lattice 'data/mlwf/lco.wout'

These vectors correspond to typical lattice vectors of a body-centered tetragonal system, that is
$$
\left( \begin{array}{c} \vec{a}_1 \\ \vec{a}_2 \\ \vec{a}_3 \end{array} \right)
=
\left( \begin{array}{ccc}
    - \frac{a}{2} &   \frac{a}{2} &   \frac{c}{2} \\
      \frac{a}{2} & - \frac{a}{2} &   \frac{c}{2} \\
      \frac{a}{2} &   \frac{a}{2} & - \frac{c}{2}
\end{array} \right)
\left( \begin{array}{c} \hat{x} \\ \hat{y} \\ \hat{z} \end{array} \right)
$$
where $a$ is the distance between two nearest atoms and $c$ is the out-of-plane lattice parameter of the conventional unit cell.

To construct the super lattice depicted in the figure above, we perform the following transformations:
$$
\left( \begin{array}{c} \vec{a}'_1 \\ \vec{a}'_2 \\ \vec{a}'_3 \end{array} \right)
=
\left( \begin{array}{ccc}
    - 1 &   1 &   2 \\
      1 & - 1 &   0 \\
      0 &   0 &   1
\end{array} \right)
\left( \begin{array}{c} \vec{a}_1 \\ \vec{a}_2 \\ \vec{a}_3 \end{array} \right)
$$
The first two vectors span the entire $xy$-plane accordingly with the figure above. To span the other planes, we need another vector that connects with a nearest plane, hence the arbitrary choice of $\vec{a}'_3$.

As for the choice of sites, again following the figure above, one site is in the original unit cell at $\vec{0}$ (blue), while the second one is at $a\hat{x}=\vec{a}_2 + \vec{a}_3$ (red).

With this information, we can now construct the obe on the super lattice which now has a correlated space that includes two sites. We use the following:

In [ ]:
sl_vectors = [( 1, 1, 2), ( 1,-1, 0), ( 1, 0, 0)] 
cluster_pts = [( 0, 0, 0), ( 0, 1, 1)]

SL  = modest.Superlattice(sl_vectors, cluster_pts)
obe_super = modest.fold(SL, obe)

dim_M  = obe_super.C_space.dim
print(dim_M)
print(obe_super.C_space)

### 🧪 Exercise 1 (optional): Play with clusters

Try to make a $2 \times 2 \times 2$ cluster.

In [ ]:
SL  = # make SuperLattice corresponding to 2x2x2 cluster
obe_super2 = # fold the original obe to the new cluster

dim_M  = # get the dimension of the C space
print(dim_M)
print(obe_super2.C_space)

## Antiferromagnetic embedding

While we start from a normal state that is paramagnetic, the magnetic order emerging from correlations is captured by the self-energy. In particular, each local site becomes polarized, *i.e* there are more electrons of a certain spin type than of the other and the self-energy of a site $0$ exhibits $\Sigma_0^\uparrow \neq \Sigma_0^\downarrow$, leading to an on-site magnetization.

In the case of an antiferromagnetic order in a Mott insulator, the direction of the on-site magnetization changes as a checkerboard pattern on the lattice, leading to the two sublattices $A$ and $B$. One natural way to treat this problem is to take our two-site super lattice and map each site to a different impurity problem and let the antiferromagnetic develop by itself. To do so, one use
> `E = make_embedding(obe_super.C_space, use_atom_equivalences=False)`

However, antiferromagnetism has the special property that the two sublattices are related by time-reversal symmetry, that is $ \Sigma_A^\sigma = \Sigma_B^\bar{\sigma}$ where given $\sigma = \uparrow$ ($\downarrow$), $\bar{\sigma} = \downarrow$ ($\uparrow$). Therefore, we can avoid solving two impurity problems at every iteration by imposing this property to the embedding object. 

**More precisely, if the sublattice $A$ is mapped to an impurity problem, the sublattice $B$ can be mapped to the exact same impurity problem, but where the spins are flipped.**

### 🧪 Exercise 2: Embedding with a flipped spin

You will now construct the required embedding in order to be able to solve the antiferromagnetic case using a single impurity with the non-trivial spin-mapping. To do so:
1. Make a default embedding out of `obe_super`. It should contain two $\alpha$-embeddings, which should be considered equivalent and mapped to the same impurity problem.
2. The embedding object contains a function called `flip_spin`. It takes as argument a list of the $\alpha$ indices for which you want to flip a spin and returns a new embedding object. Use it.

The final embedding should have two $\alpha$-blocks mapping to the same impurity `imp_idx = 0`, but where the $\alpha=0$ maps the spins $\sigma = 0, 1 \rightarrow \tau = 0, 1$ while the $\alpha = 1$ maps them as $\sigma = 0, 1 \rightarrow \tau = 1, 0$.

**In practice, this means the impurity self-energy that we solve for sublattice $A$ will be mapped to sublattice $B$ by a swap in the spin sector.**

In [ ]:
E = # make embedding from the superlattice obe
E = # flip the spin on the second self-energy (alpha =1)
print(E.description(True))

## Running the DMFT

Now that we've loaded our data, created our superlattice, and constructed our embedding, we are ready to run DMFT. As a result, we should be able to witness the build-up of the magnetization as a function of inverse temperature. The following code performs the calculation for four different temperatures, which will take a few minutes to run. Start it now so you give it some time to run. During that time, we provide some explanation.

We perform the calculations at $U=6$ eV, which is in the Mott insulating regime. In this strong coupling regime, at high temperature, the system has a metal-to-Mott insulator transition. At lower temperature, it becomes antiferromagnetic. The following calculation performs four different temperatures, increasingly small. You should observe the magnetic transition.

In order to detect magnetism, we start the calculation for each temperature by introducing a symmetry-breaking field in the self-energy, so the magnetization starts as non-zero. As it is converging, there is either (1) a suppression of the symmetry-breaking order parameter and the system restores the symmetry, characteristic of a paramagnet, or (2) a saturation of the symmetry-breaking order parameter, characteristic of a magnetic state.

Inspect the following code to make sure you understand every step. If not, search in the documentation. Once you are done, we will analyze the data.

In [ ]:
import numpy as np 
from triqs.gf import MeshImFreq, BlockGf
from triqs.operators import n
from utils.solvers import solve

betas = [3, 7, 8, 10]   # Inverse temperatures that are being computed
target_density = 2.0    # total number of electrons, 1 electron per site
U = 6.0                 # Mott insulating regime
h_int = U*n('up_0',0)*n('down_0',0) # density-denstiy interaction 
n_dmft_loops = 10       # Number of DMFT iterations

# Parameters for the TRIQS/cthyb solver
solver_params = dict(
                     length_cycle=80, 
                     n_cycles = int(1e+5), n_warmup_cycles = int(1e+3), 
                     perform_tail_fit=True, fit_min_w=15, fit_max_w=20, 
                     imag_threshold = 1e-6,
                     verbosity = 1
                     )

# We will store the magnetizations in this object, for analysis later
Magnetizations = np.zeros((len(betas), n_dmft_loops))

# BZ integration options
bz_int_opt = modest.BzIntOptions(k_grid=[4,4,4], run_adaptive=False)

for b, beta in enumerate(betas):
    
    print(f'DMFT calculation for β= {beta} 1/eV (T = {1/beta} (eV))')
    
    # Matsubara mesh
    mesh = MeshImFreq(beta, S='Fermion', n_iw=251)

    # intial self-energies (Σ = 0)
    Sigma_imp_dynamic, Sigma_imp_infty = E.make_zero_imp_self_energies(mesh)[0]

    # compute the local levels
    eps_levels = E.extract(modest.impurity_levels(obe_super))[0]
    
    for n_iter in range(n_dmft_loops):
        
        # Introducing a symmetry breaking field at the start of each iteration, to help convergence
        Sigma_imp_infty[0] += 0.1 if n_iter < 2 else 0.0
        Sigma_imp_infty[1] -= 0.1 if n_iter < 2 else 0.0

        # Embed the self-energy
        Sigma_C_dynamic, Sigma_C_infty = E.embed([Sigma_imp_dynamic], [Sigma_imp_infty])

        # Calculate the chemical potential
        mu = modest.find_chemical_potential(target_density, obe_super, 
                                            Sigma_C_dynamic, Sigma_C_infty, bz_int_opt, method="bisection", verbosity=False)

        # Calculate the local impurity levels for Δ.
        E_imp = [ block - mu for block in eps_levels]

        # Calculate the local Green's function and extract for site A
        Gloc =  E.extract(modest.gloc(obe_super, mu, Sigma_C_dynamic, Sigma_C_infty, bz_int_opt))[0]

        # Calculating the local magnetization, storing and printing
        Magnetizations[b][n_iter] = ((Gloc['up_0'].total_density()-Gloc['down_0'].total_density()).real)
        print(f"n_iter={n_iter} n↑ - n↓ = {Magnetizations[b][n_iter]}")
        
        # Compute the new Weiss field for A
        Delta_iw = modest.hybridization(E_imp, Gloc, Sigma_imp_dynamic, Sigma_imp_infty)

        # Solve the impurity model for A
        S = solve(Delta_iw, E_imp, h_int, **solver_params)

        # Extract new self-energy decomposition
        Sigma_imp_dynamic, Sigma_imp_infty = S.Sigma_dynamic, S.Sigma_Hartree

### 🧪 Exercise 3: Analyze the output

In the previous calculation, you have filled an array `Magnetizations`, one value for each iteration of each temperature. Plot the magnetization as a function of iteration for each temperature. Can you give an estimate of the critical temperature at which the system undergoes a transition to an antiferromagnet? How does the magnetization evolves with respect to temperature?

In [ ]:
from triqs.plot.mpl_interface import oplot, plt

# plot the magnetization data as a function of DMFT iteration for each beta